In [6]:
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
from src.data_loader import load_insurance_data
from src.hypothesis_tests import run_chi2_test, run_ttest

# Load clean version of data
df = load_insurance_data()

# 1. Create target indicator for Claim Frequency
df['HasClaim'] = np.where(df['TotalClaims'] > 0, 1, 0)

# 2. Create the Margin target metric
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print("Target variables engineered successfully!")
df.info()

Target variables engineered successfully!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           10000 non-null  object 
 1   Age                  10000 non-null  int64  
 2   Gender               10000 non-null  object 
 3   Province             10000 non-null  object 
 4   VehicleType          10000 non-null  object 
 5   AnnualIncome         10000 non-null  int64  
 6   RiskScore            10000 non-null  int64  
 7   AnnualPremium        10000 non-null  int64  
 8   Deductible           10000 non-null  int64  
 9   NCD                  10000 non-null  int64  
 10  PastClaims           10000 non-null  int64  
 11  Claimed              10000 non-null  bool   
 12  ClaimAmount          10000 non-null  float64
 13  TotalPremium         10000 non-null  int64  
 14  TotalClaims          10000 non-null  float64


In [7]:
import pandas as pd
import numpy as np
from src.hypothesis_tests import run_chi2_test, run_ttest

results = []

# --- Hypothesis 1: Risk Across Provinces (Frequency) ---
chi2_prov, p_prov = run_chi2_test(df, 'Province')
decision_prov = "Reject H0" if p_prov < 0.05 else "Fail to Reject H0"
results.append({
    "Hypothesis": "No risk differences across provinces", 
    "Test Used": "Chi-Squared", 
    "p-value": p_prov, 
    "Decision": decision_prov
})

# --- Hypothesis 4: Risk Between Women and Men (Frequency) ---
# Filter out unknown/null gender classifications for a clean comparison
gender_df = df[df['Gender'].isin(['Male', 'Female'])].copy()
chi2_gen, p_gen = run_chi2_test(gender_df, 'Gender')
decision_gen = "Reject H0" if p_gen < 0.05 else "Fail to Reject H0"
results.append({
    "Hypothesis": "No risk differences between Men and Women", 
    "Test Used": "Chi-Squared", 
    "p-value": p_gen, 
    "Decision": decision_gen
})

# --- Identify top two zip codes/postal codes for numerical testing ---
top_zips = df['ZipCode'].value_counts().nlargest(2).index.tolist()
zip_a_df = df[df['ZipCode'] == top_zips[0]]
zip_b_df = df[df['ZipCode'] == top_zips[1]]

# --- Hypothesis 2: Risk Between Zip Codes (Severity) ---
severity_a = zip_a_df[zip_a_df['TotalClaims'] > 0]['TotalClaims']
severity_b = zip_b_df[zip_b_df['TotalClaims'] > 0]['TotalClaims']
_, p_sev_zip = run_ttest(severity_a, severity_b)
decision_sev_zip = "Reject H0" if p_sev_zip < 0.05 else "Fail to Reject H0"
results.append({
    "Hypothesis": "No severity differences between top zip codes", 
    "Test Used": "Welch's t-test", 
    "p-value": p_sev_zip, 
    "Decision": decision_sev_zip
})

# --- Hypothesis 3: Margin Difference Between Zip Codes ---
margin_a = zip_a_df['Margin']
margin_b = zip_b_df['Margin']
_, p_marg_zip = run_ttest(margin_a, margin_b)
decision_marg_zip = "Reject H0" if p_marg_zip < 0.05 else "Fail to Reject H0"
results.append({
    "Hypothesis": "No margin differences between top zip codes", 
    "Test Used": "Welch's t-test", 
    "p-value": p_marg_zip, 
    "Decision": decision_marg_zip
})

# Display Results Summary Table
summary_table = pd.DataFrame(results)
print(summary_table.to_string(index=False))

                                   Hypothesis      Test Used  p-value          Decision
         No risk differences across provinces    Chi-Squared 0.076089 Fail to Reject H0
    No risk differences between Men and Women    Chi-Squared 0.963831 Fail to Reject H0
No severity differences between top zip codes Welch's t-test 0.894882 Fail to Reject H0
  No margin differences between top zip codes Welch's t-test 0.264234 Fail to Reject H0


-Province Risk ($p = 0.076$)Statistical Decision: Fail to Reject $H_0$.Business Meaning: Although there are slight visible variations, there is no statistically significant difference in claim frequencies across provinces.Action Item: Do not create major premium price differences based solely on the province. A blanket "Gauteng Tax" or "Western Cape Discount" is not justified by the empirical evidence.

-Gender Risk ($p = 0.964$)Statistical Decision: Fail to Reject $H_0$.Business Meaning: With a $p$-value near $1.0$, men and women in this dataset exhibit nearly identical claim frequencies. The common industry stereotype that one gender is riskier does not hold true for your target portfolio.Action Item: Marketing campaigns can target men and women equally with the same baseline pricing structures. Gender-neutral premium pricing is the most accurate approach for this portfolio.

-Zip Code Severity ($p = 0.895$) & Margin ($p = 0.264$)Statistical Decision: Fail to Reject $H_0$.Business Meaning: The average claim size (Severity) and the overall profitability per policy (Margin) do not significantly differ between your highest-volume postal codes.Action Item: Your marketing acquisition budget does not need to favor one of these top postal codes over the other. Both regions yield statistically equivalent profit margins for the business.